In [4]:
import numpy as np
import pandas as pd
import ast

In [6]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [7]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [8]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


# Data Pre-processing

In [9]:
movies.merge(credits , on='title').shape

(4809, 23)

In [10]:
movies.shape

(4803, 20)

In [11]:
credits.shape

(4803, 4)

In [12]:
movies = movies.merge(credits , on='title')

In [13]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [14]:
#  now we'll select important features for recommendation of movies and drop the
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [15]:
# we'll keep --> genres id keywords title overview cast crew
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [16]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [17]:
# check for missing n duplicate data
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [18]:
movies.dropna(inplace=True)

In [19]:
movies.duplicated().sum()

0

In [20]:
# let's see 1st entry of genre column
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [21]:
# but we only want --> [ 'Action' , 'Adventure' , 'Fantasy' , 'SciFi' ]
def object(obj):    # helper fn
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [22]:
#  but DT of above genres is string of list...so we'll first convert that into l
movies['genres'] = movies['genres'].apply(object)
movies['keywords'] = movies['keywords'].apply(object)

In [23]:
movies.head(1)
#  content is same but DT is different

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [24]:
#  we'll only consider top 3 members from cast

def convert3(obj):     # helper fn
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

In [25]:
movies['cast'] = movies['cast'].apply(convert3)

In [26]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [27]:
#  we'll only fetch director from crew

def fetch_director(obj):             # helper fn
    L=[]
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [28]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [29]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [30]:
# now adjusting overview column --> converting into list
# and removing space from single entity like : Sam Worthington --> SamWorthington

movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(' ','') for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(' ','') for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(' ','') for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(' ','') for i in x])

In [31]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


In [32]:
# now creating column tags : 
movies['tags'] = movies['overview'] + movies['genres'] + movies['cast'] + movies['crew']

In [33]:
new_df = movies[['movie_id','title','tags']]

In [34]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [35]:
# now converting list into string (under tags)
new_df['tags'].apply(lambda x:" ".join(x))

0       In the 22nd century, a paraplegic Marine is di...
1       Captain Barbossa, long believed to be dead, ha...
2       A cryptic message from Bond’s past sends him o...
3       Following the death of District Attorney Harve...
4       John Carter is a war-weary, former military ca...
                              ...                        
4804    El Mariachi just wants to play his guitar and ...
4805    A newlywed couple's honeymoon is upended by th...
4806    "Signed, Sealed, Delivered" introduces a dedic...
4807    When ambitious New York attorney Sam is sent t...
4808    Ever since the second grade when he first saw ...
Name: tags, Length: 4806, dtype: object

In [36]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

C:\Users\bhavy\AppData\Local\Temp\ipykernel_20824\3089450492.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [37]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [38]:
new_df['tags'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [39]:
# converting the complete line into lower case
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

C:\Users\bhavy\AppData\Local\Temp\ipykernel_20824\3073891067.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


In [40]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


# Text Vectorization

In [41]:
print(new_df['tags'][0])
print('\n\n')
print(new_df['tags'][1])

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction samworthington zoesaldana sigourneyweaver jamescameron



captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but nothing is quite as it seems. adventure fantasy action johnnydepp orlandobloom keiraknightley goreverbinski


In [42]:
# every movie would be represented by a vector and model will recommend closest vector(movie) on basis of
# cosine similarity (angle between them)
# text --> vectors can be done by many ways (text vectorization): text vectors, bag of words , word2vec , tfidf

# we'll use bag of words : in this we'll combine every tag , now we got a big string (tag1 + tag2 ...)
# now we've to find 5000 (say) most common words from that large text
# now check occurence of words in movies so we'll get a 5000x5000 entry table...each record is a vector

# words -->  w1  w2  w3                                                 w5000
# movies :                       <occurence>
#   m1       2    3   5                                                    0
#   m2       0    0   3                                                    2
# 
# 
# 
# 
#   m5000    1    1   1                                                    5

# now we'll remove stop words like : is are or they it can also a an to the

from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000 , stop_words='english')    # can see 5k top words via CV

In [43]:
cv.fit_transform(new_df['tags']).toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [44]:
cv.fit_transform(new_df['tags']).toarray().shape

(4806, 5000)

In [45]:
vectors =cv.fit_transform(new_df['tags']).toarray()
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [46]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [47]:
len(cv.get_feature_names_out())

5000

In [48]:
cv.get_feature_names_out()    # after stemming no repeated words shall be there

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [49]:
# these are top 5k words but it has some words with almost same meaning: actions-action , activity-activities
# using NLP (stemming)
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [50]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [51]:
ps.stem('loving')

'love'

In [52]:
# stemming 1st tag for ex
stem('In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'
)

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict samworthington zoesaldana sigourneyweav jamescameron'

In [53]:
new_df['tags'] = new_df['tags'].apply(stem)
# now repeating above steps

C:\Users\bhavy\AppData\Local\Temp\ipykernel_20824\1390429710.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [54]:
# cosine similarity ( eucledean distance is not a reliable measure in higher data )
# less theta between vectors--> more similar movies --> more cosine similarity

from sklearn.metrics.pairwise import cosine_similarity     # similarity inversly prop to distance

In [55]:
cosine_similarity(vectors)

array([[1.        , 0.15389675, 0.0860663 , ..., 0.        , 0.        ,
        0.        ],
       [0.15389675, 1.        , 0.08830216, ..., 0.03673592, 0.        ,
        0.        ],
       [0.0860663 , 0.08830216, 1.        , ..., 0.03081668, 0.        ,
        0.        ],
       ...,
       [0.        , 0.03673592, 0.03081668, ..., 1.        , 0.07897472,
        0.02746175],
       [0.        , 0.        , 0.        , ..., 0.07897472, 1.        ,
        0.05638839],
       [0.        , 0.        , 0.        , ..., 0.02746175, 0.05638839,
        1.        ]])

In [56]:
cosine_similarity(vectors).shape

(4806, 4806)

In [57]:
similarity = cosine_similarity(vectors)

In [58]:
similarity     # it is an array of arrays

array([[1.        , 0.15389675, 0.0860663 , ..., 0.        , 0.        ,
        0.        ],
       [0.15389675, 1.        , 0.08830216, ..., 0.03673592, 0.        ,
        0.        ],
       [0.0860663 , 0.08830216, 1.        , ..., 0.03081668, 0.        ,
        0.        ],
       ...,
       [0.        , 0.03673592, 0.03081668, ..., 1.        , 0.07897472,
        0.02746175],
       [0.        , 0.        , 0.        , ..., 0.07897472, 1.        ,
        0.05638839],
       [0.        , 0.        , 0.        , ..., 0.02746175, 0.05638839,
        1.        ]])

In [59]:
similarity[0]     # telling similarity of 1st movie with other

array([1.        , 0.15389675, 0.0860663 , ..., 0.        , 0.        ,
       0.        ])

In [60]:
sorted(similarity[0] , reverse=True)

[1.0000000000000002,
 0.26967994498529685,
 0.25819888974716115,
 0.24913643956121986,
 0.2480694691784169,
 0.2439750182371333,
 0.24152294576982397,
 0.22360679774997896,
 0.22360679774997894,
 0.22056438662814232,
 0.21926450482675733,
 0.21693045781865616,
 0.21693045781865616,
 0.21693045781865616,
 0.21213203435596426,
 0.21213203435596426,
 0.2076136996343499,
 0.20732210721568234,
 0.2051956704170308,
 0.2051956704170308,
 0.2051956704170308,
 0.20412414523193148,
 0.20377787840885372,
 0.20225995873897262,
 0.20080483222562473,
 0.19999999999999998,
 0.1976423537605237,
 0.19518001458970663,
 0.19518001458970663,
 0.19518001458970663,
 0.19364916731037088,
 0.19069251784911845,
 0.1889822365046136,
 0.1889822365046136,
 0.1889822365046136,
 0.18650096164806276,
 0.18650096164806276,
 0.18428853505018536,
 0.18257418583505539,
 0.18257418583505539,
 0.18257418583505539,
 0.18257418583505539,
 0.18257418583505539,
 0.18257418583505539,
 0.1813690625275029,
 0.17928429140015906,


In [61]:
list(enumerate(similarity[0]))   # so created index position wrt distances (list of tuples)

[(0, 1.0000000000000002),
 (1, 0.1538967528127731),
 (2, 0.08606629658238704),
 (3, 0.05923488777590924),
 (4, 0.12048289933537483),
 (5, 0.15339299776947407),
 (6, 0.035355339059327376),
 (7, 0.22056438662814232),
 (8, 0.08944271909999159),
 (9, 0.11180339887498947),
 (10, 0.1632993161855452),
 (11, 0.11338934190276817),
 (12, 0.16269784336399212),
 (13, 0.07071067811865475),
 (14, 0.17213259316477408),
 (15, 0.07669649888473704),
 (16, 0.1224744871391589),
 (17, 0.13801311186847084),
 (18, 0.06030226891555272),
 (19, 0.11028219331407116),
 (20, 0.07961173386514128),
 (21, 0.08451542547285165),
 (22, 0.10846522890932808),
 (23, 0.15),
 (24, 0.08606629658238704),
 (25, 0.02795084971874737),
 (26, 0.18257418583505539),
 (27, 0.16137430609197573),
 (28, 0.1732050807568877),
 (29, 0.09784921095801634),
 (30, 0.11028219331407116),
 (31, 0.21693045781865616),
 (32, 0.0659380473395787),
 (33, 0.158113883008419),
 (34, 0.0),
 (35, 0.17928429140015906),
 (36, 0.1976423537605237),
 (37, 0.08770

In [62]:
sorted(list(enumerate(similarity[0])),reverse=True)

[(4805, 0.0),
 (4804, 0.0),
 (4803, 0.0),
 (4802, 0.0),
 (4801, 0.036760731104690386),
 (4800, 0.0),
 (4799, 0.08770580193070293),
 (4798, 0.03015113445777636),
 (4797, 0.0),
 (4796, 0.0),
 (4795, 0.0),
 (4794, 0.0),
 (4793, 0.0),
 (4792, 0.0),
 (4791, 0.0),
 (4790, 0.04152273992686998),
 (4789, 0.0),
 (4788, 0.0),
 (4787, 0.0),
 (4786, 0.0),
 (4785, 0.0),
 (4784, 0.0),
 (4783, 0.0),
 (4782, 0.0),
 (4781, 0.044721359549995794),
 (4780, 0.0),
 (4779, 0.0),
 (4778, 0.0),
 (4777, 0.0),
 (4776, 0.0),
 (4775, 0.043852900965351466),
 (4774, 0.0),
 (4773, 0.0),
 (4772, 0.03492151478847891),
 (4771, 0.0),
 (4770, 0.0),
 (4769, 0.0),
 (4768, 0.0),
 (4767, 0.0512989176042577),
 (4766, 0.0),
 (4765, 0.0),
 (4764, 0.0),
 (4763, 0.0),
 (4762, 0.0),
 (4761, 0.045643546458763846),
 (4760, 0.0),
 (4759, 0.035355339059327376),
 (4758, 0.0),
 (4757, 0.06454972243679029),
 (4756, 0.0),
 (4755, 0.0),
 (4754, 0.0),
 (4753, 0.0),
 (4752, 0.0),
 (4751, 0.0),
 (4750, 0.0),
 (4749, 0.0),
 (4748, 0.037267799624

In [63]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])

[(0, 1.0000000000000002),
 (151, 0.26967994498529685),
 (4405, 0.25819888974716115),
 (942, 0.24913643956121986),
 (1440, 0.2480694691784169),
 (1192, 0.2439750182371333),
 (3606, 0.24152294576982397),
 (495, 0.22360679774997896),
 (4348, 0.22360679774997894),
 (7, 0.22056438662814232),
 (972, 0.21926450482675733),
 (31, 0.21693045781865616),
 (94, 0.21693045781865616),
 (4008, 0.21693045781865616),
 (2997, 0.21213203435596426),
 (3728, 0.21213203435596426),
 (973, 0.2076136996343499),
 (1214, 0.20732210721568234),
 (61, 0.2051956704170308),
 (68, 0.2051956704170308),
 (322, 0.2051956704170308),
 (1342, 0.20412414523193148),
 (4046, 0.20377787840885372),
 (208, 0.20225995873897262),
 (199, 0.20080483222562473),
 (466, 0.19999999999999998),
 (36, 0.1976423537605237),
 (232, 0.19518001458970663),
 (483, 0.19518001458970663),
 (1612, 0.19518001458970663),
 (83, 0.19364916731037088),
 (1654, 0.19069251784911845),
 (1392, 0.1889822365046136),
 (1804, 0.1889822365046136),
 (2405, 0.188982236

In [64]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])[1:6]

[(151, 0.26967994498529685),
 (4405, 0.25819888974716115),
 (942, 0.24913643956121986),
 (1440, 0.2480694691784169),
 (1192, 0.2439750182371333)]

In [65]:
similarity[0].shape

(4806,)

# Main Function

In [66]:
def recommend(movie):       # main fn : takes a movie as an argument and find its index and would see similarity[i] and return top 5 movies on basis of top 5 similarities
    movie_index = new_df[new_df['title']==movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    
    for i in movies_list:
            print(new_df.iloc[i[0]].title)

In [67]:
new_df.iloc[2999].title

'The Boy'

In [68]:
recommend('Avatar')

Beowulf
The Helix... Loaded
The Book of Life
Krull
Small Soldiers


In [ ]:
import pickle
pickle.dump(new_df.to_dict(),open('movie_dict.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))

In [71]:
import pickle
import numpy as np

similarity = pickle.load(open('similarity.pkl', 'rb'))
similarity = similarity.astype(np.float32)

# Save with highest compression
import gzip
with gzip.open('similarity.pkl', 'wb', compresslevel=9) as f:
    pickle.dump(similarity, f)

import os
size = os.path.getsize('similarity.pkl') / (1024*1024)
print(f"New size: {size:.1f} MB")

New size: 29.3 MB


In [73]:
import gzip
import streamlit as st

@st.cache_data
def load_data():
    movies_dict = pickle.load(open('movie_dict.pkl', 'rb'))
    movies = pd.DataFrame(movies_dict)
    with gzip.open('similarity.pkl', 'rb') as f:
        similarity = pickle.load(f)
    return movies, similarity

2026-05-05 16:30:19.869 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


# Deployment